In [15]:
# Install packages
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns

#Set file paths
root_fldr = Path.cwd().parent
raw_fldr = root_fldr/"data"/"raw"
processed_fldr = root_fldr/'data'/'processed'

#Load iNaturalist data and convert coordinates to points (AI was used here to help figure out converting points in a GeoDataFrame)
raw_invasives = raw_fldr/"inaturalist_invasives.csv"
invasive_coords = pd.read_csv(raw_fldr/"inaturalist_invasives.csv")
invasives_gdf = gpd.GeoDataFrame(invasive_coords, geometry=gpd.points_from_xy(invasive_coords["longitude"], invasive_coords["latitude"]),
    crs="EPSG:4326"
)
invasives_gdf.head()

,id,uuid,observed_on_string,observed_on,created_at,updated_at,quality_grade,license,url,image_url,...,oauth_application_id,place_guess,latitude,longitude,species_guess,scientific_name,common_name,iconic_taxon_name,taxon_id,geometry
0,2579,903406bb-3f9a-4465-b82a-2a8eb0394796,5/9/2009 11:06,5/9/2009,5/9/2009 15:05,5/30/2020 16:27,research,CC0,http://www.inaturalist.org/observations/2579,https://inaturalist-open-data.s3.amazonaws.com...,...,NaN,"Lake Temescal Regional Recreation Area, Oaklan...",37.844845,-122.229355,yellow flag,Iris pseudacorus,Yellow Iris,Plantae,47779,POINT (-122.22935 37.84484)
1,3184,d9a90ba1-b8b8-4970-bac0-7cd7e2983388,8/2/2009 16:54,8/2/2009,8/10/2009 14:48,12/10/2016 21:57,research,NaN,http://www.inaturalist.org/observations/3184,https://inaturalist-open-data.s3.amazonaws.com...,...,NaN,"Santa Clara County, US-CA, US",37.243595,-121.877426,Yellow Starthistle,Centaurea solstitialis,Yellow Star-Thistle,Plantae,52588,POINT (-121.87743 37.2436)
2,4151,5bee462e-49e7-45f6-a35b-c530a2e0d02b,12/31/2009 16:59,12/31/2009,1/9/2010 9:26,6/1/2020 13:48,research,NaN,http://www.inaturalist.org/observations/4151,https://static.inaturalist.org/photos/8367/med...,...,NaN,"Almaden Quicksilver Park, CA, US",37.205551,-121.879127,poison hemlock,Conium maculatum,poison hemlock,Plantae,52998,POINT (-121.87913 37.20555)
3,7583,5b9f5daa-4720-4bba-bdd3-767ef88838e6,6/12/2010,6/12/2010,6/16/2010 18:45,6/2/2020 15:54,research,CC-BY,http://www.inaturalist.org/observations/7583,https://inaturalist-open-data.s3.amazonaws.com...,...,NaN,"Sonoma, CA",38.302235,-122.455406,Yellow Starthistle,Centaurea solstitialis,Yellow Star-Thistle,Plantae,52588,POINT (-122.45541 38.30223)
4,7589,d1503aa4-9f75-4cd9-9dd0-e3b925a55add,6/12/2010,6/12/2010,6/17/2010 18:38,8/13/2019 23:19,research,CC-BY,http://www.inaturalist.org/observations/7589,https://inaturalist-open-data.s3.amazonaws.com...,...,NaN,"Sonoma, CA",38.301674,-122.456612,Tree of heaven,Ailanthus altissima,tree-of-heaven,Plantae,57278,POINT (-122.45661 38.30167)


In [16]:
#List Columns
invasives_gdf.columns

Index(['id', 'uuid', 'observed_on_string', 'observed_on', 'created_at',
       'updated_at', 'quality_grade', 'license', 'url', 'image_url',
       'sound_url', 'tag_list', 'description', 'num_identification_agreements',
       'num_identification_disagreements', 'captive_cultivated',
       'oauth_application_id', 'place_guess', 'latitude', 'longitude',
       'species_guess', 'scientific_name', 'common_name', 'iconic_taxon_name',
       'taxon_id', 'geometry'],
      dtype='str')

In [17]:
#Check for missing values
invasives_gdf.isna().mean().sort_values()*100

id                                    0.000000
uuid                                  0.000000
observed_on_string                    0.000000
observed_on                           0.000000
created_at                            0.000000
updated_at                            0.000000
quality_grade                         0.000000
url                                   0.000000
captive_cultivated                    0.000000
image_url                             0.000000
num_identification_disagreements      0.000000
num_identification_agreements         0.000000
scientific_name                       0.000000
common_name                           0.000000
latitude                              0.000000
longitude                             0.000000
iconic_taxon_name                     0.000000
taxon_id                              0.000000
geometry                              0.000000
place_guess                           0.132948
species_guess                        15.434184
oauth_applica

Columns with over 90% missing values are not useful for analysis, so we will not analyze them. Many of these columns provide information about the quality and specifics of iNaturalist observations in the program; we need data on the flora species observed. Columns providing details on the actual flora species were kept, with coordinates being removed since the addition of the geometry column, and repetitive columns like observed_on and observed_on_string were removed.

In [18]:
#Cleaning
inaturalist_gdf_cleaned = (
    invasives_gdf
    .loc[:, ['id', 'observed_on','place_guess', 'geometry', 'scientific_name', 'common_name'], ]
    .rename(columns = {
            "observed_on" : "observation_date",
            "place_guess" : "location",
            })
    .set_crs("EPSG:4326", allow_override=True)
        .to_crs("EPSG:3310")
)

inaturalist_gdf_cleaned['observation_date'] = pd.to_datetime(inaturalist_gdf_cleaned['observation_date'], format='%m/%d/%Y')

In [19]:
#save to processed folder
inaturalist_gdf_cleaned.to_file(
    processed_fldr / "inaturalist_gdf_cleaned.gpkg",
    driver="GPKG"
)

# AI consultation: save as a gpkg so the geometry variable is retained

In [20]:
inaturalist_gdf_cleaned = gpd.read_file(processed_fldr / "inaturalist_gdf_cleaned.gpkg")

inaturalist_gdf_cleaned.head()

,id,observation_date,location,scientific_name,common_name,geometry
0,2579,2009-05-09,"Lake Temescal Regional Recreation Area, Oaklan...",Iris pseudacorus,Yellow Iris,POINT (-195897.48 -16764.107)
1,3184,2009-08-02,"Santa Clara County, US-CA, US",Centaurea solstitialis,Yellow Star-Thistle,POINT (-166300.728 -84258.436)
2,4151,2009-12-31,"Almaden Quicksilver Park, CA, US",Conium maculatum,poison hemlock,POINT (-166535.23 -88483.577)
3,7583,2010-06-12,"Sonoma, CA",Centaurea solstitialis,Yellow Star-Thistle,POINT (-214440.087 34555.771)
4,7589,2010-06-12,"Sonoma, CA",Ailanthus altissima,tree-of-heaven,POINT (-214546.951 34496.183)
